In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
import scanpy as sc
import scipy.sparse as sp
import numpy as np
from TrainAE import fit_gene_cell_autoencoders
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import glorot, uniform
from torch_geometric.utils import softmax
import copy
import random
from collections import defaultdict
from torch_geometric.utils import negative_sampling
import math
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
from TrainHGT import build_hgt_model, train_hgt_with_subgraph_sampling, infer_hgt_embeddings_with_global_graph

In [2]:
import torch

print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

0
NVIDIA A40


#### AE部分

In [3]:
adata_sub = sc.read_h5ad("/home/user/jiayuran/Omics/Adenomyosis/Data/ProcessedGSE218044/sc_adenomyosis_hvg_1000cells_by_major_celltype.h5ad")
# 只保留高变基因
hvg_mask = adata_sub.var["var.features"].to_numpy(dtype=bool)
print("HVG数量:", hvg_mask.sum())
adata_sub = adata_sub[:, hvg_mask].copy()
print("HVG后维度:", adata_sub.shape)
print(adata_sub)

HVG数量: 2000
HVG后维度: (1000, 2000)
AnnData object with n_obs × n_vars = 1000 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'group', 'percent.mt', 'RNA_snn_res.0.8', 'cluster', 'major_celltype', 'ident'
    var: 'var.features'
    obsm: 'HARMONY', 'X_pca', 'X_umap'
    layers: 'logcounts'


In [4]:
# 取表达矩阵
X = adata_sub.X

# sparse -> dense
if sp.issparse(X):
    X = X.toarray()
else:
    X = np.asarray(X)

# 转成 gene x cell
gene_cell = X.T.astype(np.float32)

print("gene_cell shape:", gene_cell.shape)   # [n_genes, n_cells]

gene_cell shape: (2000, 1000)


In [5]:
result = fit_gene_cell_autoencoders(
    gene_cell=gene_cell,
    latent_dim=256,
    hidden_dims=(512,),
    dropout=0.0,
    output_activation=None,
    lr=1e-3,
    weight_decay=0.0,
    batch_size=1024,
    max_epochs=100,
    val_ratio=0.1,
    patience=30,
)

gene_embedding = result["gene_embedding"]
cell_embedding = result["cell_embedding"]

print(gene_embedding.shape)  # [n_genes, 256]
print(cell_embedding.shape)  # [n_cells, 256]

Training gene AE...
Epoch 000 | train_loss=24.912970 | val_loss=20.920395
Epoch 020 | train_loss=4.059146 | val_loss=8.688055
Epoch 040 | train_loss=2.141335 | val_loss=6.735942
Epoch 060 | train_loss=1.645080 | val_loss=6.104274
Epoch 080 | train_loss=1.226840 | val_loss=5.823875
Epoch 099 | train_loss=1.039592 | val_loss=5.651136
cuda:0
Training cell AE...
Epoch 000 | train_loss=19.313313 | val_loss=83.946579
Epoch 020 | train_loss=6.918314 | val_loss=59.915703
Epoch 040 | train_loss=3.785817 | val_loss=7.876215
Epoch 060 | train_loss=2.547186 | val_loss=3.996675
Epoch 080 | train_loss=2.012054 | val_loss=3.269424
Epoch 099 | train_loss=1.863952 | val_loss=3.299822
cuda:0
(2000, 256)
(1000, 256)


In [6]:
gene_names = adata_sub.var_names.to_numpy()
cell_names = adata_sub.obs_names.to_numpy()

print(len(gene_names), len(cell_names))
print(gene_embedding.shape, cell_embedding.shape)

2000 1000
(2000, 256) (1000, 256)


#### GAT+HGT训练

In [7]:
train_result = train_hgt_with_subgraph_sampling(
    gene_cell=gene_cell,
    feature_mode="ae",
    loss_type="kl",
    gene_embedding=gene_embedding,
    cell_embedding=cell_embedding,
    n_hid=64,
    n_heads=4,
    n_layers=2,
    dropout=0.2,
    lr=1e-3,
    weight_decay=0.0,
    max_epochs=5,
    patience=10,
    n_batch=30, # number of subgraph batches per epoch 如果把seed_gene_batch_size、sample_width_gene_to_cell、sample_width_cell_to_gene三个参数调大了，建议反而把它调小，因为大 batch 子图已经更有信息量，不需要那么多小子图重复训练，每个 epoch 的 Python/采样开销会明显下降
    seed_gene_batch_size=64,   # Size of the batch of genes to sample in each iteration 每个 batch 覆盖更多 seed genes 单步更“值钱”
    sample_depth=2,
    sample_width_gene_to_cell=32,   # 每个 seed gene 连接的 cell 数量，过大过小都不好，过大噪声多，过小信息不足 每个 seed gene 扩展到更多 cells gene-cell 关系学得更充分
    sample_width_cell_to_gene=32,   # 每个 seed cell 连接的 gene 数量，过大过小都不好，过大噪声多，过小信息不足 每个 seed cell 扩展到更多 genes gene-cell 关系学得更充分
    use_gat=True,
    knn_k=5,
    knn_metric="cosine",
    gat_hidden_dim=128,
    gat_heads=4,
    gat_dropout=0.2,
    verbose=True,
)

Building global gene-gene KNN graph...
Building global cell-cell KNN graph...
Global gg edges: 16108, Global cc edges: 7112


Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Start epoch 000


Batch 000:   0%|          | 0/30 [00:00<?, ?it/s]

  epoch 000 batch 0/30
  epoch 000 batch 10/30
  epoch 000 batch 20/30
Epoch 000 | avg_loss=10504.648551
Start epoch 001


...

  epoch 002 batch 0/30
  epoch 002 batch 10/30
  epoch 002 batch 20/30
Epoch 002 | avg_loss=5973.267594
Start epoch 003


Batch 003:   0%|          | 0/30 [00:00<?, ?it/s]

  epoch 003 batch 0/30
  epoch 003 batch 10/30
  epoch 003 batch 20/30
Epoch 003 | avg_loss=5839.010628
Start epoch 004


Batch 004:   0%|          | 0/30 [00:00<?, ?it/s]

  epoch 004 batch 0/30
  epoch 004 batch 10/30
  epoch 004 batch 20/30
Epoch 004 | avg_loss=5739.670182


#### 计算gene-cell score 矩阵

In [9]:
#### 推理
infer_result = infer_hgt_embeddings_with_global_graph(
    model=train_result["model"],
    gene_cell=gene_cell,
    gene_embedding=gene_embedding,
    cell_embedding=cell_embedding,
    global_edge_index_gg=train_result["global_edge_index_gg"],
    global_edge_index_cc=train_result["global_edge_index_cc"],
    gene_block_size=1024
)

refined_gene_embedding = infer_result["refined_gene_embedding"]
refined_cell_embedding = infer_result["refined_cell_embedding"]

In [10]:
#### 计算重构矩阵 就是gene-cell score矩阵
S_gc = refined_gene_embedding @ refined_cell_embedding.T

# cell-wise z-score
S_gc_z = (S_gc - S_gc.mean(axis=0, keepdims=True)) / (
    S_gc.std(axis=0, keepdims=True) + 1e-6
)
print(S_gc_z.shape)   # [n_genes, n_cells]

# softmax
S_shift = S_gc - S_gc.max(axis=0, keepdims=True)
S_gc_soft = np.exp(S_shift) / np.exp(S_shift).sum(axis=0, keepdims=True)
print(S_gc_soft.shape)   # [n_genes, n_cells]

(2000, 1000)
(2000, 1000)


#### 依据通路计算得分

In [15]:
hallmark_apoptosis = [
    "BAX",
    "BAK1",
    "BCL2",
    "BCL2L1",
    "CASP3",
    "CASP7",
    "CASP8",
    "CASP9",
    "FAS",
    "FASLG",
    "TNF",
    "APAF1",
    "CYCS",
    "PARP1",
    "BID",
    "BAD",
    "PMAIP1",
    "BBC3",
]

# adata_sub.var_names 就是当前基因顺序
gene_names = adata_sub.var_names.astype(str).tolist()

# 建一个 gene -> index 的字典
gene_to_idx = {g: i for i, g in enumerate(gene_names)}

hallmark_apoptosis = [
    "BAX", "BAK1", "BCL2", "BCL2L1", "CASP3", "CASP7", "CASP8", "CASP9",
    "FAS", "FASLG", "TNF", "APAF1", "CYCS", "PARP1", "BID", "BAD",
    "PMAIP1", "BBC3"
]

# 找到存在的基因及其索引
present_genes = [g for g in hallmark_apoptosis if g in gene_to_idx]
missing_genes = [g for g in hallmark_apoptosis if g not in gene_to_idx]
seed_idx = [gene_to_idx[g] for g in present_genes]

print("存在的基因:", present_genes)
print("缺失的基因:", missing_genes)
print("seed_idx:", seed_idx)
print("存在基因数量:", len(seed_idx))

存在的基因: ['TNF', 'PMAIP1']
缺失的基因: ['BAX', 'BAK1', 'BCL2', 'BCL2L1', 'CASP3', 'CASP7', 'CASP8', 'CASP9', 'FAS', 'FASLG', 'APAF1', 'CYCS', 'PARP1', 'BID', 'BAD', 'BBC3']
seed_idx: [716, 1782]
存在基因数量: 2


In [16]:
# seed_idx = pathway gene indices 用Z-score矩阵里对应行的平均值作为这个pathway的score
pathway_score_Zscore = S_gc_z[seed_idx].mean(axis=0)

# Softmax
pathway_score_Softmax = S_gc_soft[seed_idx].sum(axis=0)

In [18]:
pathway_score_Zscore

array([4.12876606e-02, 5.09701073e-02, 6.43006712e-02, 4.44538146e-02,
       6.47380948e-02, 3.31769139e-02, 5.78070730e-02, 4.12124097e-02,
       5.96384853e-02, 8.32383484e-02, 8.68486017e-02, 4.29828018e-02,
       3.49587053e-02, 5.68686575e-02, 5.34433424e-02, 2.78094172e-01,
       1.62542313e-02, 3.53554189e-02, 4.55416590e-02, 8.38058442e-02,
       1.96680576e-02, 3.12358886e-02, 3.48537415e-02, 3.60575467e-02,
       8.63662362e-03, 5.34633845e-02, 4.59403396e-02, 1.27897486e-01,
       6.08887374e-02, 6.71594441e-02, 4.73159105e-02, 6.71744049e-02,
       5.33071458e-02, 8.45697373e-02, 7.92135298e-02, 1.15834773e-02,
       1.75126940e-02, 7.44208395e-02, 8.13600719e-02, 9.59102064e-02,
       7.19747990e-02, 5.24935722e-02, 3.91545445e-02, 1.27309904e-01,
       4.96164709e-02, 5.33234626e-02, 1.43742114e-01, 3.21924239e-02,
       7.90853351e-02, 6.09956086e-02, 5.87975234e-02, 3.00861001e-02,
       4.72251326e-02, 7.70219266e-02, 2.52369940e-02, 9.20751691e-02,
      